# 02 — Create PASS/FAIL Labels

## Objective
Convert the training set's COCO defect annotations into one image-level classification label per image.

## Inputs
- `../data/raw/train/_annotations.coco.json`: image records, category definitions, and annotated defect regions.

## Method
1. Read the COCO JSON file and identify annotated image IDs.
2. Map category IDs to their readable defect names.
3. Associate each image with its recorded defect types and count.
4. Apply the provisional rule **one or more annotations → FAIL; no annotations → PASS**.
5. Inspect the resulting label counts and class proportions.

## Outputs
- In-memory `labels_df` with `image_id`, `filename`, `number_of_defects`, `defect_types`, and `label`.
- Printed class counts and proportions. **This notebook does not export a label CSV.**

## Important assumption
An image without an annotation is treated as PASS. This reflects the supplied annotation records; it is not independent proof that the package is defect-free. Label accuracy should be checked against visual inspection and the project's acceptance criteria.


Create Pass Fail Labels

## 1. Dependencies
Install pandas if it is not already available in the notebook environment.

In [1]:
# Notebook dependency: pandas provides the DataFrame used for the label table.
%pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Imports
Load the JSON parser, portable path handling, and pandas for tabular results.

In [2]:
# Import utilities for reading the COCO JSON file and organising image labels.
import json
from pathlib import Path
import pandas as pd

## 3. Read the COCO annotations
Load annotation data and collect the distinct IDs of images with recorded defects. The set avoids counting the same image more than once.

In [3]:
# The training split supplies image metadata and defect annotations.
annotation_path = Path("../data/raw/train/_annotations.coco.json")

# Parse the JSON structure into Python dictionaries and lists.
with open(annotation_path, "r") as file:
    coco_data = json.load(file)
# Track unique images with at least one annotation; duplicates are removed by the set.
annotated_image_ids = set()

for annotation in coco_data["annotations"]:
    # Each annotation points back to its parent image through image_id.
    annotated_image_ids.add(annotation["image_id"])

## 4. Build image-level labels
First translate category IDs into defect names and group annotations by image. Then label each recorded image as FAIL when it has at least one annotated defect, otherwise PASS.

In [4]:
# Build reusable dictionaries keyed by COCO category and image identifiers.
# Create a lookup from category ID to category name
category_names = {}

for category in coco_data["categories"]:
    # Convert numeric category IDs to their readable defect names.
    category_names[category["id"]] = category["name"]


# Count annotations per image
annotation_count = {}

# Record the number of annotations associated with each image.

for annotation in coco_data["annotations"]:
    image_id = annotation["image_id"]

    if image_id not in annotation_count:
        annotation_count[image_id] = 0

    annotation_count[image_id] += 1


# Store defect types for each image
defect_types = {}

for annotation in coco_data["annotations"]:
    # Associate the recorded defect names with the corresponding image.
    image_id = annotation["image_id"]
    category_id = annotation["category_id"]
    defect_name = category_names[category_id]

    if image_id not in defect_types:
        defect_types[image_id] = []

    defect_types[image_id].append(defect_name)


# Build PASS / FAIL label table
image_labels = []

for image_record in coco_data["images"]:

    # Assemble one classification record for every image in the COCO image list.
    image_id = image_record["id"]
    filename = image_record["file_name"]

    defects = defect_types.get(image_id, [])
    number_of_defects = len(defects)

    # An empty defect list means no recorded annotation, not independently verified quality.
    if number_of_defects > 0:
        label = "FAIL"
    else:
        # Apply the notebook’s image-level labelling assumption.
        label = "PASS"

    image_labels.append({
        "image_id": image_id,
        "filename": filename,
        # Retain filename and defect metadata alongside the binary target.
        "number_of_defects": number_of_defects,
        "defect_types": defects,
        "label": label
    })

labels_df = pd.DataFrame(image_labels)

labels_df.head()

,image_id,filename,number_of_defects,defect_types,label
0,0,frame_with_red_8214_jpg.rf.5acda82172bcf022bd7...,1,[packaging-error],FAIL
1,1,frame_with_red_1237_jpg.rf.7c920cb9b9d7c993551...,1,[unsealed],FAIL
2,2,frame_with_red_6884_jpg.rf.73d4cccf385f78c6473...,1,[unsealed],FAIL
3,3,frame_with_red_2737_jpg.rf.8af748cd8904c67c7a0...,0,[],PASS
4,4,frame_with_red_10465_jpg.rf.800ed5e3f0775757ce...,1,[wrinkle],FAIL


## 5. Preview the label table
Create or display the dataframe containing one row per image. This repeats a dataframe construction already present at the end of the preceding cell; it is retained unchanged.

In [5]:
# Keep the original dataframe creation and preview step.
labels_df = pd.DataFrame(image_labels)

labels_df.head()

,image_id,filename,number_of_defects,defect_types,label
0,0,frame_with_red_8214_jpg.rf.5acda82172bcf022bd7...,1,[packaging-error],FAIL
1,1,frame_with_red_1237_jpg.rf.7c920cb9b9d7c993551...,1,[unsealed],FAIL
2,2,frame_with_red_6884_jpg.rf.73d4cccf385f78c6473...,1,[unsealed],FAIL
3,3,frame_with_red_2737_jpg.rf.8af748cd8904c67c7a0...,0,[],PASS
4,4,frame_with_red_10465_jpg.rf.800ed5e3f0775757ce...,1,[wrinkle],FAIL


## 6. Check class balance
Show both the number and proportion of PASS and FAIL images. A skewed distribution can make classification accuracy misleading.

In [6]:
# Absolute counts show how many samples are available for each class.
print(labels_df["label"].value_counts())
# Relative frequencies show class imbalance independent of dataset size.
print(labels_df["label"].value_counts(normalize=True))

label
FAIL    2665
PASS     684
Name: count, dtype: int64
label
FAIL    0.79576
PASS    0.20424
Name: proportion, dtype: float64


## 7. Display class counts
Repeat the counts as a pandas Series for a concise notebook output.

In [7]:
# Summarise image-level label frequencies again.
labels_df["label"].value_counts()

label
FAIL    2665
PASS     684
Name: count, dtype: int64